# Road Damage Assessment — Colab PRO Setup

Notebook này chạy **Streamlit web dashboard** trên Google Colab PRO (GPU) + tạo public URL qua cloudflared tunnel.

## ⚠️ Cấu hình Runtime type (BẮT BUỘC trước khi chạy)

**Menu > Runtime > Change runtime type:**
- **Runtime type:** Python 3
- **Hardware accelerator:** `T4 GPU` ⚠️ (KHÔNG chọn CPU — inference sẽ chậm 20x)
- **High-RAM:** `On` ✅ (cần cho video dài + batch ảnh)
- **Runtime version:** Latest (recommended)

> Lý do chọn T4: quota Colab Pro cao nhất (16GB VRAM, đủ cho YOLOv12s + FastSAM). A100/H100 thường quota thấp/greyed out.

**Cảnh báo:** Colab session tạm thời (idle timeout ~90 phút, giới hạn 24h). Luôn có bản backup local (`streamlit run app.py` trên laptop).

**Ưu tiên dùng Colab cho:**
- GPU inference nhanh (T4 ~30-50ms thay vì 896ms CPU)
- Render video demo dài (GPU xử lý 17967 frame trong vài phút)
- Train T3 (YOLOv12s-seg trên RDD2022)

**Debug:** Tab `📜 Console` trong dashboard hiển thị log runtime + system info (CUDA, GPU memory).

## 1. Cài đặt dependencies

In [ ]:
# Install Streamlit + cloudflared + ML deps
!pip install streamlit cloudflared ultralytics opencv-python-headless supervision Pillow matplotlib huggingface-hub -q

# Verify GPU (T4 from Runtime type config)
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    gpu_name = torch.cuda.get_device_name(0)
    if 'T4' in gpu_name:
        print('✅ T4 GPU detected — optimal for Colab Pro')
    elif 'A100' in gpu_name:
        print('✅ A100 GPU detected — premium (may have lower quota)')
    elif 'L4' in gpu_name:
        print('✅ L4 GPU detected — newer, fast')
    else:
        print(f'⚠️ Unknown GPU: {gpu_name}')
else:
    print('❌ WARNING: No GPU! Go to Runtime > Change runtime type > Hardware accelerator: T4 GPU')
    print('   Also enable High-RAM toggle for video processing')

## 2. Tải source code + model weights

**Option A:** Upload folder `adaptive/` + `dumps/` lên Google Drive, rồi mount.

**Option B:** Git clone (nếu đã push lên GitHub).

In [ ]:
# Option A: Mount Google Drive (recommended if you have files there)
from google.colab import drive
drive.mount('/content/drive')

# Adjust paths to your Drive structure
# Example: if you copied 'New folder' to My Drive/
import os
DRIVE_BASE = '/content/drive/MyDrive/New folder'  # CHANGE THIS
ADAPTIVE = f'{DRIVE_BASE}/adaptive'
DUMPS = f'{DRIVE_BASE}/dumps'

print(f'Adaptive exists: {os.path.exists(ADAPTIVE)}')
print(f'Dumps exists: {os.path.exists(DUMPS)}')
print(f'Model exists: {os.path.exists(f"{DUMPS}/models/yolo-rdd2022-benchmark/yolo12s_seed0_best.pt")}')

In [ ]:
# Option B: Git clone from GitHub (if pushed)
# !git clone https://github.com/YOUR_USERNAME/road-damage-assessment.git /content/repo
# ADAPTIVE = '/content/repo/adaptive'
# DUMPS = '/content/repo/dumps'

In [ ]:
# Set DUMPS_ROOT env var (engine_bridge auto-detects, but explicit is safer)
import os, sys
os.chdir(ADAPTIVE)
sys.path.insert(0, ADAPTIVE)
os.environ['DUMPS_ROOT'] = DUMPS  # tell engine_bridge where dumps/ is

# Test import — engine_bridge._find_dumps_root() will use DUMPS_ROOT env var
import engine_bridge as eb
print(f'Dumps root: {eb._DUMPS_ROOT}')
print(f'Model exists: {eb.DEFAULT_MODEL_PATH.exists()}')
print(f'PCI data exists: {eb.DEFAULT_PCI_DATA.exists()}')
print(f'Samples count: {len(list(eb.DUMPS_SAMPLES.glob("*.jpg")))}')

## 3. Chạy Streamlit + cloudflared tunnel

Chạy cell dưới → đợi ~10s → click URL public (xxxx.trycloudflare.com).

In [ ]:
# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared installed')

In [ ]:
# Run Streamlit in background
import subprocess, os
os.chdir(ADAPTIVE)

# Kill any existing streamlit
!pkill -f streamlit 2>/dev/null; sleep 1

# Start streamlit (background, headless, GPU enabled)
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
print(f'Streamlit PID: {proc.pid}')

# Wait for streamlit to start
import time
time.sleep(5)
print('Streamlit started on port 8501')

In [ ]:
# Start cloudflared tunnel → get public URL
import subprocess, re, time

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Capture the URL from stderr (cloudflared prints it there)
url = None
for _ in range(30):
    line = tunnel.stderr.readline()
    if line:
        print(line.strip())
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    time.sleep(1)

if url:
    print(f'\n=== PUBLIC URL ===')
    print(f'🌐 {url}')
    print(f'\nMở URL này trong browser. Dashboard chạy trên GPU Colab.')
    print(f'URL sống cho đến khi Colab session kết thúc.')
else:
    print('URL not found in cloudflared output. Check stderr above.')

## 4. (Tùy chọn) Render video demo với GPU

GPU Colab render video nhanh gấp 20x CPU. Dùng cho video dài.

In [ ]:
# Render video demo với GPU (cuda)
import os
os.chdir(ADAPTIVE)

# Upload video của bạn vào Colab trước, hoặc dùng video mẫu
INPUT_VIDEO = f'{DUMPS}/runs/detect/outputs/images/ultralytics_test/0.avi'
OUTPUT_VIDEO = f'{ADAPTIVE}/outputs/demo_colab.mp4'

os.makedirs(f'{ADAPTIVE}/outputs', exist_ok=True)

# Run with GPU + process all frames (stride=1, no cap)
!python render_video.py \
#    --input "{INPUT_VIDEO}" \
#    --output "{OUTPUT_VIDEO}" \
#    --device cuda \
#    --stride 5 \
#    --max-frames 200 \
#    --confidence 0.15

In [ ]:
# Download rendered video
from google.colab import files
if os.path.exists(OUTPUT_VIDEO):
    files.download(OUTPUT_VIDEO)
else:
    print(f'{OUTPUT_VIDEO} not found. Check render output above.')

## 5. Dừng & dọn dẹp

Khi xong, dừng Streamlit + tunnel để giải phóng Colab.

In [ ]:
# Stop everything
!pkill -f streamlit 2>/dev/null
!pkill -f cloudflared 2>/dev/null
print('Stopped Streamlit + cloudflared tunnel')

## Troubleshooting

- **URL không hiện:** Chạy lại cell cloudflared. Đôi khi cần 2-3 lần.
- **Streamlit lỗi import:** Verify `engine_bridge.py` đã patch đường dẫn Colab đúng.
- **Model not found:** Download weights: `!python {DUMPS}/scripts/download_model.py`
- **No GPU:** Runtime > Change runtime type > GPU (Colab PRO)
- **Session ngắt:** Colab PRO có idle timeout ~90 phút. Tương tác thường xuyên hoặc giữ tab active.

## Plan B — Local backup

Nếu Colab ngắt giữa demo:
```bash
# Trên laptop
cd D:\Antigravity\New folder\adaptive
D:\Antigravity\New folder\dumps\.venv\Scripts\python.exe -m streamlit run app.py
```
Mở http://localhost:8501. Inference CPU ~896ms nhưng tin cậy.